<a href="https://colab.research.google.com/github/rudalshan0412-code/attention-is-all-you-need-pytorch/blob/main/04)_postion_wide_feedforward.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Drive 연결

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 프로젝트 경로 설정

from pathlib import Path
import sys

PROJECT_ROOT = Path("/content/drive/MyDrive/attention_is_all_you_need")
SRC_DIR = PROJECT_ROOT / "src"

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)

PROJECT_ROOT: /content/drive/MyDrive/attention_is_all_you_need
SRC_DIR: /content/drive/MyDrive/attention_is_all_you_need/src


In [ ]:
# 기존 파일 확인

for path in sorted(SRC_DIR.glob("*.py")):
    print(path.name)

attention.py
encoder.py
encoder_layer.py
feed_forward.py
multi_head_attention.py
positional_encoding.py


In [ ]:
'''
Position wide feedforward는 Encoder, Decoder에 존재하는 sub-layer중 하나로 단어와 단어 사이의 정보 교류가 없다.
대신, 이미 교류된 정보들의 가중치를 재계산함으로서 정보를 재정렬한다.
'''

'\nPosition wide feedforward는 Encoder, Decoder에 존재하는 sub-layer중 하나로 단어와 단어 사이의 정보 교류가 없다.\n대신, 이미 교류된 정보들의 가중치를 재계산함으로서 정보를 재정렬한다.\n'

In [ ]:
# nn.Linear 가 마지막 차원에 적용되는 것 확인

import torch
import torch.nn as nn

batch_size = 2
seq_len = 3
d_model = 4
d_ff = 8

x = torch.randn( # 평균 0, 표준 편차 1인 tensor(()는 생성할 tensor의 형태)
    batch_size,
    seq_len,
    d_model,
)

linear1 = nn.Linear(d_model, d_ff) # 마지막 차원을 d_model로 받은 다음 d_ff로 내보냄

hidden = linear1(x)

print("x shape      :", x.shape)
print("hidden shape :", hidden.shape) # d_model이 d_ff로 바뀐 것을 확인 할 수 있음

x shape      : torch.Size([2, 3, 4])
hidden shape : torch.Size([2, 3, 8])


In [ ]:
# position끼리 섞이지 않는지 직접 확인

torch.manual_seed(0)

x = torch.randn(1, 2, 4)

linear = nn.Linear(4, 8)

whole_output = linear(x) # shape는 (1, 2, 8)

token0_output = linear(x[0, 0, :]) # 0번째 batch, 0번째 토큰의 :(전부)를 지정 -> shape는 (4, ) -> linear 하기에 8개로 나
token1_output = linear(x[0, 1, :])

print("전체 Tensor를 한 번에 계산:")
print(whole_output)

print("\nToken 0만 따로 계산:")
print(token0_output)

print("\nToken 1만 따로 계산:")
print(token1_output)

print("\nToken 0 동일:",
      torch.allclose(whole_output[0, 0], token0_output)) # 결과상으로는 동일한데 동일함 결과는 False임. why..?

print("Token 1 동일:",
      torch.allclose(whole_output[0, 1], token1_output))

전체 Tensor를 한 번에 계산:
tensor([[[ 0.6096,  0.5769, -1.0700,  0.7194,  0.2615, -0.8567,  0.0013,
          -1.2962],
         [-1.1749, -0.5236,  0.1271,  1.2989,  0.3385,  0.0760,  0.3081,
           0.8308]]], grad_fn=<ViewBackward0>)

Token 0만 따로 계산:
tensor([ 0.6096,  0.5769, -1.0700,  0.7194,  0.2615, -0.8567,  0.0013, -1.2962],
       grad_fn=<ViewBackward0>)

Token 1만 따로 계산:
tensor([-1.1749, -0.5236,  0.1271,  1.2989,  0.3385,  0.0760,  0.3081,  0.8308],
       grad_fn=<ViewBackward0>)

Token 0 동일: False
Token 1 동일: True


In [ ]:
# 전체 FFN Shape 확인

import torch
import torch.nn as nn
import torch.nn.functional as F

batch_size = 2
seq_len = 4
d_model = 8
d_ff = 32

x = torch.randn(
    batch_size,
    seq_len,
    d_model,
)

linear1 = nn.Linear(d_model, d_ff)
linear2 = nn.Linear(d_ff, d_model)

hidden = linear1(x)
activated = F.relu(hidden) # relu를 적용해도 shape는 유지
output = linear2(activated)

print("입력 x shape           :", x.shape)
print("첫 번째 Linear 이후    :", hidden.shape)
print("ReLU 이후             :", activated.shape)
print("두 번째 Linear 이후    :", output.shape)
print("최종 output shape      :", output.shape)

입력 x shape           : torch.Size([2, 4, 8])
첫 번째 Linear 이후    : torch.Size([2, 4, 32])
ReLU 이후             : torch.Size([2, 4, 32])
두 번째 Linear 이후    : torch.Size([2, 4, 8])
최종 output shape      : torch.Size([2, 4, 8])


In [ ]:
# PositionwiseFeedForward 전체 코드

import torch.nn as nn
import torch.nn.functional as F


class PositionwiseFeedForward(nn.Module):
    """
    Position-wise Feed Forward Network.

    Input:
        (batch_size, seq_len, d_model)

    Output:
        (batch_size, seq_len, d_model)
    """

    def __init__(self, d_model, d_ff):
        super().__init__()

        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = self.linear1(x)
        x = F.relu(x)
        x = self.linear2(x)

        return x

In [ ]:
# src/feed_forward.py 저장

%%writefile /content/drive/MyDrive/attention_is_all_you_need/src/feed_forward.py

import torch.nn as nn
import torch.nn.functional as F


class PositionwiseFeedForward(nn.Module):
    """
    Position-wise Feed Forward Network.

    Input:
        (batch_size, seq_len, d_model)

    Output:
        (batch_size, seq_len, d_model)
    """

    def __init__(self, d_model, d_ff):
        super().__init__()

        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = self.linear1(x)
        x = F.relu(x)
        x = self.linear2(x)

        return x

Overwriting /content/drive/MyDrive/attention_is_all_you_need/src/feed_forward.py


In [ ]:
# 파일구조 다시 확인

for path in sorted(SRC_DIR.glob("*.py")):
    print(path.name)

attention.py
encoder.py
encoder_layer.py
feed_forward.py
multi_head_attention.py
positional_encoding.py


In [ ]:
# 저장한 class import

from src.feed_forward import PositionwiseFeedForward

print(PositionwiseFeedForward)

<class 'src.feed_forward.PositionwiseFeedForward'>


In [ ]:
# 기본 shape 테스트

import torch

batch_size = 2
seq_len = 3
d_model = 4
d_ff = 8

model = PositionwiseFeedForward(
    d_model=d_model,
    d_ff=d_ff,
)

x = torch.randn(
    batch_size,
    seq_len,
    d_model,
)

output = model(x)

print("input shape :", x.shape)
print("output shape:", output.shape)

input shape : torch.Size([2, 3, 4])
output shape: torch.Size([2, 3, 4])


In [ ]:
# class 내부 shape 직접 확인

hidden = model.linear1(x)
activated = F.relu(hidden)
final_output = model.linear2(activated)

print("입력 x:")
print(x.shape)

print("\n첫 번째 Linear:")
print(hidden.shape)

print("\nReLU:")
print(activated.shape)

print("\n두 번째 Linear:")
print(final_output.shape)

print("\nmodel(x):")
print(output.shape)

입력 x:
torch.Size([2, 3, 4])

첫 번째 Linear:
torch.Size([2, 3, 8])

ReLU:
torch.Size([2, 3, 8])

두 번째 Linear:
torch.Size([2, 3, 4])

model(x):
torch.Size([2, 3, 4])


In [ ]:
# d_model = 8, d_ff = 32 shape 테스트

batch_size = 2
seq_len = 4
d_model = 8
d_ff = 32

model = PositionwiseFeedForward(
    d_model=d_model,
    d_ff=d_ff,
)

x = torch.randn(
    batch_size,
    seq_len,
    d_model,
)

hidden = model.linear1(x)
activated = F.relu(hidden)
output = model.linear2(activated)

print("입력 x shape        :", x.shape)
print("Linear 1 이후       :", hidden.shape)
print("ReLU 이후           :", activated.shape)
print("Linear 2 이후       :", output.shape)

입력 x shape        : torch.Size([2, 4, 8])
Linear 1 이후       : torch.Size([2, 4, 32])
ReLU 이후           : torch.Size([2, 4, 32])
Linear 2 이후       : torch.Size([2, 4, 8])


In [ ]:
# batch_size 에 종속되지 않는지 확인

model = PositionwiseFeedForward(
    d_model=8,
    d_ff=32,
)

x = torch.randn(
    5,
    4,
    8,
)

output = model(x)

print("input :", x.shape)
print("output:", output.shape)

input : torch.Size([5, 4, 8])
output: torch.Size([5, 4, 8])


In [ ]:
# seq_len에 종속되지 않는지 확인

x = torch.randn(
    2,
    10,
    8,
)

output = model(x)

print("input :", x.shape)
print("output:", output.shape)

input : torch.Size([2, 10, 8])
output: torch.Size([2, 10, 8])


In [ ]:
# Position-wise 동작 직접 검증

torch.manual_seed(0)

model = PositionwiseFeedForward(
    d_model=4,
    d_ff=8,
)

x = torch.randn(
    1,
    2,
    4,
)

whole_output = model(x)

token0_output = model(x[0, 0, :])
token1_output = model(x[0, 1, :])

print("전체 입력 output:")
print(whole_output)

print("\nToken 0 단독 output:")
print(token0_output)

print("\nToken 1 단독 output:")
print(token1_output)

print(
    "\nToken 0 동일:",
    torch.allclose(
        whole_output[0, 0],
        token0_output,
    ),
)

print(
    "Token 1 동일:",
    torch.allclose(
        whole_output[0, 1],
        token1_output,
    ),
)

전체 입력 output:
tensor([[[0.1847, 0.2188, 0.3807, 0.3307],
         [0.2037, 0.2209, 0.3547, 0.2781]]], grad_fn=<ViewBackward0>)

Token 0 단독 output:
tensor([0.1847, 0.2188, 0.3807, 0.3307], grad_fn=<ViewBackward0>)

Token 1 단독 output:
tensor([0.2037, 0.2209, 0.3547, 0.2781], grad_fn=<ViewBackward0>)

Token 0 동일: True
Token 1 동일: True


In [ ]:
# 최종 통합 테스트

import torch
import torch.nn.functional as F

from src.feed_forward import PositionwiseFeedForward


torch.manual_seed(0)

batch_size = 2
seq_len = 4
d_model = 8
d_ff = 32

model = PositionwiseFeedForward(
    d_model=d_model,
    d_ff=d_ff,
)

x = torch.randn(
    batch_size,
    seq_len,
    d_model,
)


# 1. 전체 FFN 실행
output = model(x)


# 2. 중간 Shape 확인
hidden = model.linear1(x)
activated = F.relu(hidden)
manual_output = model.linear2(activated)


assert x.shape == (
    batch_size,
    seq_len,
    d_model,
)

assert hidden.shape == (
    batch_size,
    seq_len,
    d_ff,
)

assert activated.shape == (
    batch_size,
    seq_len,
    d_ff,
)

assert output.shape == (
    batch_size,
    seq_len,
    d_model,
)


# 3. forward()와 직접 계산 결과 비교
assert torch.allclose(
    output,
    manual_output,
)


# 4. ReLU 확인
negative_mask = hidden < 0

assert torch.all(
    activated[negative_mask] == 0
)


# 5. Position-wise 동작 확인
single_token_output = model(
    x[0, 0, :]
)

assert torch.allclose(
    output[0, 0],
    single_token_output,
)


# 6. 다른 batch / sequence 길이 확인
x2 = torch.randn(
    3,
    7,
    d_model,
)

output2 = model(x2)

assert output2.shape == (
    3,
    7,
    d_model,
)


print("모든 Position-wise FFN 테스트 통과!")
print()
print("입력 shape       :", x.shape)
print("Linear 1 shape  :", hidden.shape)
print("ReLU shape      :", activated.shape)
print("Linear 2 shape  :", output.shape)

모든 Position-wise FFN 테스트 통과!

입력 shape       : torch.Size([2, 4, 8])
Linear 1 shape  : torch.Size([2, 4, 32])
ReLU shape      : torch.Size([2, 4, 32])
Linear 2 shape  : torch.Size([2, 4, 8])
